In [ ]:
# 1. Проверка GPU
!nvidia-smi

# 2. Установка утилит и Ollama
!sudo apt-get update -qq && sudo apt-get install -y -qq zstd pciutils
!curl -fsSL https://ollama.com/install.sh | sh

In [ ]:
import os
import subprocess
import time

os.environ["OLLAMA_HOST"] = "0.0.0.0:11434"
os.environ["OLLAMA_ORIGINS"] = "*"

print("Запускаем сервер Ollama...")
subprocess.Popen(['ollama', 'serve'], stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
time.sleep(8)

# Проверка готовности API
!curl -s http://localhost:11434/api/tags

In [ ]:
!ollama pull gemma3:12b

In [ ]:
proxy_code = r'''
import json
import httpx
from fastapi import FastAPI, Request, Response
from fastapi.responses import StreamingResponse
import uvicorn

app = FastAPI()

@app.api_route("/{path:path}", methods=["GET", "POST"])
async def proxy(request: Request, path: str):
    body = await request.body()
    target = f"http://localhost:11434/{path}"

    # Логируем входящий запрос от ПК
    if body and path in ("api/chat", "api/generate"):
        try:
            data = json.loads(body)
            if "messages" in data:
                print(f"\n>>> [PC → LLM] user: {data['messages'][-1].get('content','')!r}", flush=True)
            elif "prompt" in data:
                print(f"\n>>> [PC → LLM] prompt: {data['prompt']!r}", flush=True)
        except Exception as e:
            print(f"\n>>> [PC → LLM] (parse error: {e})", flush=True)

    client = httpx.AsyncClient(timeout=300)
    req = client.build_request(
        method=request.method,
        url=target,
        content=body,
        headers={"Content-Type": "application/json"},
    )
    resp = await client.send(req, stream=True)
    ctype = resp.headers.get("content-type", "")

    # Стриминговый ответ (Ollama отдаёт application/x-ndjson)
    if "x-ndjson" in ctype or "event-stream" in ctype:
        async def gen():
            full = ""
            try:
                async for chunk in resp.aiter_bytes():
                    for line in chunk.split(b"\n"):
                        if not line.strip():
                            continue
                        try:
                            obj = json.loads(line)
                            tok = obj.get("message", {}).get("content", "") or obj.get("response", "")
                            if tok:
                                full += tok
                            if obj.get("done"):
                                print(f"<<< [LLM → PC] answer: {full!r}\n", flush=True)
                        except Exception:
                            pass
                    yield chunk
            finally:
                await resp.aclose()
                await client.aclose()

        return StreamingResponse(gen(), media_type=ctype)

    # Обычный (не-стриминговый) ответ
    try:
        content = await resp.aread()
        try:
            data = json.loads(content)
            ans = data.get("message", {}).get("content", "") or data.get("response", "")
            if ans:
                print(f"<<< [LLM → PC] answer: {ans!r}\n", flush=True)
        except Exception:
            pass
        return Response(content=content, status_code=resp.status_code, media_type="application/json")
    finally:
        await resp.aclose()
        await client.aclose()

if __name__ == "__main__":
    uvicorn.run(app, host="0.0.0.0", port=11435, log_level="warning")
'''

with open("/content/log_proxy.py", "w", encoding="utf-8") as f:
    f.write(proxy_code)

import subprocess, time
subprocess.Popen(["python", "/content/log_proxy.py"])
time.sleep(5)
print("✓ Прокси с логированием запущен на :11435")

In [ ]:
import os
import re
import subprocess
import time

if not os.path.exists("./cloudflared"):
    !wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
    !chmod +x cloudflared

print("Запускаем Cloudflare Tunnel...")
tunnel_process = subprocess.Popen(
    ['./cloudflared', 'tunnel', '--url', 'http://localhost:11435'],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
)

time.sleep(10)
public_url = None
for line in tunnel_process.stdout:
    m = re.search(r'https://[a-zA-Z0-9-]+\.trycloudflare\.com', line)
    if m:
        public_url = m.group(0)
        break

if public_url:
    print("\n" + "=" * 60)
    print(f"URL для OLLAMA_BASE_URL: {public_url}")
    print("=" * 60)
else:
    print("Не удалось получить URL. Перезапустите эту ячейку.")

In [ ]:
import subprocess, time

# Останавливаем старый процесс прокси, если был запущен
!pkill -f "log_proxy.py" 2>/dev/null || true
time.sleep(1)

with open("/content/log_proxy.py", "w", encoding="utf-8") as f:
    f.write(proxy_code)

log_file = open("/content/proxy.log", "w", encoding="utf-8")
subprocess.Popen(
    ["python", "-u", "/content/log_proxy.py"],
    stdout=log_file,
    stderr=subprocess.STDOUT,
)
time.sleep(3)
print("✓ Прокси запущен на :11435, логи пишутся в /content/proxy.log")

In [ ]:
!tail -f /content/proxy.log